In [ ]:
import pandas as pd
import os
import shutil
import json
import getpass


from sqlalchemy import create_engine
from sqlalchemy.engine import URL
# Local src

from src.df_overview import df_overview
from src.generate_metadata import generate_metadata
from src.create_database import create_database


# Create database and variables for database connection

password = getpass.getpass('Enter database password: ')
db_name = 'customer_rfm_retail'
base_url = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",          # change if needed
    password=password,
    host="localhost",
    port=5432,
    database="postgres"          
)

engine = create_engine(base_url)

create_database(base_url=base_url, db_name=db_name, password=password)

print('Done')

Done


In [ ]:
path_base_csv = '../data/02_processed/base_retail.csv'

df = pd.read_csv(path_base_csv)

In [ ]:
df_overview(df)

================================= Head =================================
   customer_id   last_purchase_date  frequency  monetary  recency  r_score  \
0        17592  2009-12-01 10:49:00          1    148.30    738.0        1   
1        17818  2009-12-02 11:34:00          1    124.78    737.0        1   
2        17087  2009-12-02 10:41:00          1    221.53    737.0        1   
3        17056  2009-12-01 12:55:00          1    128.60    737.0        1   
4        13526  2009-12-01 13:13:00          2   1182.00    737.0        1   

   f_score  m_score  
0        1        1  
1        1        1  
2        1        1  
3        1        1  
4        3        3  
================================= Tail =================================
      customer_id   last_purchase_date  frequency  monetary  recency  r_score  \
5842        12985  2011-12-09 10:46:00          2   1239.38      0.0        5   
5843        14056  2011-12-08 13:36:00         23   8152.71      0.0        5   
5844      

In [ ]:
password = getpass.getpass('Enter database password: ')
base_url = 'localhost:5432'
db_name = 'customer_rfm_retail'

create_database(base_url=base_url, db_name=db_name)

In [ ]:
df["recency_score"] = pd.qcut(
    df["recency"], 
    5, 
    labels=[5,4,3,2,1], 
    duplicates='drop'
)

df["frequency_score"] = pd.qcut(
    df["frequency"].rank(method="first"), 
    5, 
    labels=[1,2,3,4,5], 
    duplicates='drop'
)

df["monetary_score"] = pd.qcut(
    df["monetary"], 
    5, 
    labels=[1,2,3,4,5], 
    duplicates='drop'
)

df["recency_score"] = df["recency_score"].astype(int)
df["frequency_score"] = df["frequency_score"].astype(int)
df["monetary_score"] = df["monetary_score"].astype(int)

df["R_F_Score"] = df["recency_score"].astype(str) + df["frequency_score"].astype(str)

seg_map = {
    r'[1-2][1-2]': 'hibernating',
    r'[1-2][3-4]': 'at_risk',
    r'[1-2]5': 'cant_lose',
    r'3[1-2]': 'about_to_sleep',
    r'33': 'need_attention',
    r'[3-4][4-5]': 'loyal_customers',
    r'41': 'promising',
    r'51': 'new_customers',
    r'[4-5][2-3]': 'potential_loyalists',
    r'5[4-5]': 'champions'
}

df["segment"] = df["R_F_Score"].replace(seg_map, regex = True)

In [ ]:
df['sales_only'] = df['revenue'].clip(lower=0)
df['returns_only'] = df['revenue'].clip(upper=0).abs()

ratios = df.groupby('customer_id').agg(
    total_sales=('sales_only', 'sum'),
    total_returns=('returns_only', 'sum')
)

ratios['return_ratio'] = ratios['total_returns'] / ratios['total_sales']
ratios['return_ratio'] = ratios['return_ratio'].fillna(0)
ratios.reset_index()
ratios
df = df.set_index('customer_id')
df = df.merge(ratios[['return_ratio']], left_index=True, right_index=True, how='left')

In [ ]:
df_overview(df)

================================= Head =================================
              last_purchase_date  frequency  monetary  recency  r_score  \
customer_id                                                               
17592        2009-12-01 10:49:00          1    148.30    738.0        1   
17818        2009-12-02 11:34:00          1    124.78    737.0        1   
17087        2009-12-02 10:41:00          1    221.53    737.0        1   
17056        2009-12-01 12:55:00          1    128.60    737.0        1   
13526        2009-12-01 13:13:00          2   1182.00    737.0        1   

             f_score  m_score  recency_score  frequency_score  monetary_score  \
customer_id                                                                     
17592              1        1              1                1               1   
17818              1        1              1                1               1   
17087              1        1              1                1               1

In [ ]:
pd_to_csv_path = os.path.join('..', 'data', '03_featured', 'featured_rfm_retail.csv')

os.makedirs(os.path.dirname(pd_to_csv_path), exist_ok=True)
df.to_csv(pd_to_csv_path, index=False)

generate_metadata(
    df=df,
    csv_path=pd_to_csv_path
)

print(f"Saved to: {pd_to_csv_path}")

Metadata saved: ../data/03_featured/featured_rfm_retail_metadata.json
Saved to: ../data/03_featured/featured_rfm_retail.csv


customer_age_days = snapshot_date - first_purchase_date
avg_interpurchase_time = customer_age_days / (frequency - 1)
avg_order_value